# AP1 — análise quantitativa e visual dos resultados da Fase 0

Este notebook recalcula os resultados a partir dos CSVs e logs brutos. Os caminhos são relativos à raiz do repositório e os arquivos de entrada são somente lidos.

In [1]:
from pathlib import Path
import sys, hashlib, re, math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

def find_root(start=Path.cwd()):
    for p in (start, *start.parents):
        if (p / "data" / "records.csv").exists() and (p / "docs").exists(): return p
    raise FileNotFoundError("Raiz BeeCooler não encontrada")

ROOT = find_root()
DATA = ROOT / "data"
FIG = ROOT / "figuras" / "resultados"
DOC = ROOT / "docs" / "AP1_RESULTADOS_ANALISE.md"
FIG.mkdir(parents=True, exist_ok=True)
SESSIONS = {
    "A": "20260904T091350-0300_444e0683",
    "B": "20260916T102940-0300_58ad6dab",
}
WINDOW_SECONDS = 10
THERMAL_CROP_MIN = None  # Ex.: (5, 25); None usa a sessão inteira
EXPECTED_FRAGMENTS = 7
INPUTS = {
    "records.csv": DATA / "records.csv", "batches.csv": DATA / "batches.csv",
    "gateway_stats.csv": DATA / "gateway_stats.csv",
    "log_A": DATA / f"gateway_raw_{SESSIONS['A']}.log",
    "log_B": DATA / f"gateway_raw_{SESSIONS['B']}.log",
}
AUDIT_HASHES = {
 "records.csv":"3c9ba8a2accaeffe37c1b398a57b4d80fcd199808dc4d1310e985ca73dbf304f",
 "batches.csv":"fd6d203508f8d012c45377c856f5b5dd84ca7c3bc984535b60d6b7757c509a63",
 "gateway_stats.csv":"f3ee9f78580f8e5b4631bb8080edcc9388f8d20710036143c9373e31c35530d1",
 "log_A":"4a3807a47d34fad28990b70dbcce0301c3dfbeadfb09563b0259016cd7150999",
 "log_B":"c40f3d782c9e8ee8904ef8531fa36dd7e566ceecd03911098d8376a03f2b1241",
}
def sha256(path): return hashlib.sha256(path.read_bytes()).hexdigest()
HASH_BEFORE = {k:sha256(v) for k,v in INPUTS.items()}
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}")
print(pd.DataFrame({"sha256":HASH_BEFORE, "confere_auditoria":{k:HASH_BEFORE[k]==AUDIT_HASHES[k] for k in INPUTS}}))

Python 3.13.5 | pandas 3.0.6 | numpy 2.5.3 | matplotlib 3.11.2
                                                              sha256  \
records.csv        3c9ba8a2accaeffe37c1b398a57b4d80fcd199808dc4d1...   
batches.csv        fd6d203508f8d012c45377c856f5b5dd84ca7c3bc98453...   
gateway_stats.csv  f3ee9f78580f8e5b4631bb8080edcc9388f8d207100361...   
log_A              4a3807a47d34fad28990b70dbcce0301c3dfbeadfb0956...   
log_B              c40f3d782c9e8ee8904ef8531fa36dd7e566ceecd03911...   

                   confere_auditoria  
records.csv                     True  
batches.csv                     True  
gateway_stats.csv               True  
log_A                           True  
log_B                           True  


In [2]:
# Leitura com preservação inicial dos campos vazios; conversão numérica explícita.
raw_records = pd.read_csv(INPUTS["records.csv"], dtype=str, keep_default_na=False)
raw_batches = pd.read_csv(INPUTS["batches.csv"], dtype=str, keep_default_na=False)
raw_stats = pd.read_csv(INPUTS["gateway_stats.csv"], dtype=str, keep_default_na=False)
num_records = ["batch_id","sample_index","sensor_ms","elapsed_ms","internal_temp_c","internal_rh_pct","external_temp_c","external_rh_pct","accel_x_g","accel_y_g","accel_z_g","accel_magnitude_g"]
num_batches = ["batch_id","batch_start_ms","bytes","records","interval_ms","duplicates"]
num_stats = [c for c in raw_stats if c not in ("session_id","received_at")]
def numeric(df, cols):
    out=df.copy()
    for c in cols: out[c]=pd.to_numeric(out[c], errors="coerce")
    out["received_at"]=pd.to_datetime(out["received_at"], errors="coerce")
    return out
records, batches, stats = numeric(raw_records,num_records), numeric(raw_batches,num_batches), numeric(raw_stats,num_stats)
print("Linhas totais:", len(records), len(batches), len(stats))
print("Campos vazios físicos (antes da conversão):")
print(raw_records.replace("", np.nan).isna().sum()[lambda x:x>0])

Linhas totais: 31029 518 517
Campos vazios físicos (antes da conversão):
Series([], dtype: int64)


In [3]:
# Parser dos logs: timestamp, evento e pares chave=valor.
def parse_log(path, label):
    rows=[]
    for line_no,line in enumerate(path.read_text().splitlines(),1):
        m=re.match(r"^(\S+)\s+(\S+)(?:\s+(.*))?$", line)
        if not m: continue
        ts,event,tail=m.groups(); d={"line_no":line_no,"session":label,"received_at":pd.to_datetime(ts),"event":event,"raw":line}
        for k,v in re.findall(r"(\w+)=([^\s]+)",tail or ""):
            d[k]=v
        rows.append(d)
    return pd.DataFrame(rows)
logs=pd.concat([parse_log(INPUTS[f"log_{s}"],s) for s in SESSIONS],ignore_index=True)
logs["batch"] = pd.to_numeric(logs.get("batch_id"), errors="coerce")
logs["seq"] = pd.to_numeric(logs.get("sequence_id"), errors="coerce")
logs["status"] = logs.get("result")
frag = logs.get("fragment", pd.Series(index=logs.index, dtype=str)).fillna("").str.extract(r"(?P<frag>\d+)/(?P<total>\d+)")
logs["frag"] = pd.to_numeric(frag["frag"], errors="coerce")
logs["total"] = pd.to_numeric(frag["total"], errors="coerce")
print(logs.groupby(["session","event"]).size().unstack(fill_value=0))

event    ACK_TX  BATCH_COMPLETE  BATCH_REPLACED  FRAGMENT_RX  GATEWAY_STATS  \
session                                                                       
A          1216             168              12         1216            168   
B           462              66               0          462             66   

event    RECORD  SEQUENCE_GAP  
session                        
A         10080            18  
B          3960             0  


In [4]:
# Funções de qualidade, estatística e comunicação.
vars_ = {"T interna (°C)":"internal_temp_c","T externa (°C)":"external_temp_c","RH interna (%)":"internal_rh_pct","RH externa (%)":"external_rh_pct","X (g)":"accel_x_g","Y (g)":"accel_y_g","Z (g)":"accel_z_g","Magnitude (g)":"accel_magnitude_g","ΔT (°C)":"delta_t","ΔRH (p.p.)":"delta_rh"}
def fmt(v, d=3): return "—" if pd.isna(v) else f"{v:.{d}f}"
def interval_summary(s):
    s=np.asarray(s.dropna(),float)
    return {"n":len(s),"mín":np.min(s) if len(s) else np.nan,"mediana":np.median(s) if len(s) else np.nan,"máx":np.max(s) if len(s) else np.nan}
def session_data(label):
    sid=SESSIONS[label]
    r=records[records.session_id.eq(sid)].copy().sort_values(["batch_id","sample_index"])
    b=batches[batches.session_id.eq(sid)].copy().sort_values("batch_id")
    st=stats[stats.session_id.eq(sid)].copy().sort_values("received_at")
    l=logs[logs.session.eq(label)].copy().sort_values("received_at")
    r["tempo_min"]=(r.sensor_ms-r.sensor_ms.min())/60000
    r["delta_t"]=r.internal_temp_c-r.external_temp_c; r["delta_rh"]=r.internal_rh_pct-r.external_rh_pct
    r["mag_calc"]=np.sqrt(r.accel_x_g**2+r.accel_y_g**2+r.accel_z_g**2)
    r["mag_error"]=r.mag_calc-r.accel_magnitude_g
    # Janela por trechos sem lacuna temporal maior que 0,75 s.
    r=r.sort_values("sensor_ms").copy(); r["block"]=(r.sensor_ms.diff().fillna(0)>750).cumsum()
    window_n=max(2,round(WINDOW_SECONDS*1000/r.sensor_ms.diff().median()))
    r["mag_std_10s"]=r.groupby("block").accel_magnitude_g.transform(lambda x:x.rolling(window_n,min_periods=max(3,window_n//2)).std())
    return r,b,st,l,window_n
def comm_metrics(label,l):
    rx=l[l.event.eq("FRAGMENT_RX")].copy(); rx["batch_id"]=rx["batch"]; rx["sequence_id"]=rx["seq"]
    unique=rx[rx.get("status","").eq("UNIQUE")].copy(); dup=rx[rx.get("status","").eq("DUPLICATE")].copy()
    lo,hi=int(rx.batch.min()),int(rx.batch.max()); allb=pd.DataFrame({"batch_id":range(lo,hi+1)})
    counts=unique.groupby("batch").size().rename("unique"); allb=allb.join(counts,on="batch_id").fillna({"unique":0}); allb.unique=allb.unique.astype(int)
    allb["state"]=np.select([allb.unique.eq(7),allb.unique.eq(0)],["completo","sem fragmento"],default="parcial")
    seq=sorted(unique.seq.dropna().astype(int).unique()); seqlo,seqhi=min(seq),max(seq); missing=sorted(set(range(seqlo,seqhi+1))-set(seq))
    gaps=rx.received_at.dropna().sort_values().diff().dt.total_seconds(); silence=pd.DataFrame({"inicio":rx.received_at.dropna().sort_values().iloc[:-1].to_numpy(),"fim":rx.received_at.dropna().sort_values().iloc[1:].to_numpy(),"segundos":gaps.iloc[1:].to_numpy()}).sort_values("segundos",ascending=False)
    return {"rx":rx,"unique":unique,"dup":dup,"batches":allb,"seqlo":seqlo,"seqhi":seqhi,"missing":missing,"silence":silence,"coverage":len(seq)/(seqhi-seqlo+1)}
D={}; C={}; QUALITY=[]; SENSOR_STATS=[]
for label in SESSIONS:
    r,b,st,l,wn=session_data(label); D[label]=(r,b,st,l,wn); C[label]=comm_metrics(label,l)
    within=r.groupby("batch_id").sensor_ms.diff(); starts=b.batch_start_ms.diff();
    quality={"Sessão":label,"registros":len(r),"batches CSV":len(b),"batches com 60":int((r.groupby("batch_id").size().eq(60)).sum()),"NaN sensores":int(r[["internal_temp_c","external_temp_c","internal_rh_pct","external_rh_pct"]].isna().sum().sum()),"duplicatas de registro":int(r.duplicated(["batch_id","sample_index"]).sum()),"início sensor":r.sensor_ms.min(),"fim sensor":r.sensor_ms.max(),"duração sensor (s)":(r.sensor_ms.max()-r.sensor_ms.min())/1000,"início recepção":r.received_at.min(),"fim recepção":r.received_at.max(),"duração recepção (s)":(r.received_at.max()-r.received_at.min()).total_seconds(),"batch min–max":f"{int(b.batch_id.min())}–{int(b.batch_id.max())}","sequence min–max":f"{C[label]['seqlo']}–{C[label]['seqhi']}","sensor=start+elapsed":bool(np.isclose(r.sensor_ms, r.batch_id.map(b.set_index('batch_id').batch_start_ms)+r.elapsed_ms, equal_nan=True).all())}
    QUALITY.append(quality)
    for name,col in vars_.items():
        x=r[col].dropna(); SENSOR_STATS.append({"Sessão":label,"variável":name,"n válido":len(x),"mínimo":x.min(),"média":x.mean(),"mediana":x.median(),"máximo":x.max(),"desvio padrão":x.std(),"amplitude":x.max()-x.min()})
quality=pd.DataFrame(QUALITY); sensor_stats=pd.DataFrame(SENSOR_STATS)
print(quality.to_string(index=False))
print("Distribuições de sample_index:")
for s in SESSIONS: print(s, D[s][0].sample_index.value_counts().sort_index().to_dict())
print("Estatísticas dos sensores:"); display(sensor_stats.round(4))

Sessão  registros  batches CSV  batches com 60  NaN sensores  duplicatas de registro  início sensor  fim sensor  duração sensor (s)                  início recepção                     fim recepção  duração recepção (s) batch min–max sequence min–max  sensor=start+elapsed
     A      10080          168             168             0                       0         448012     7531554            7083.542 2026-09-04 09:14:14.163000-03:00 2026-09-04 11:11:49.103000-03:00              7054.940        16–234         106–1638                  True
     B       3960           66              66             2                       0           3940     1964112            1960.172 2026-09-16 10:29:52.759000-03:00 2026-09-16 11:02:06.648000-03:00              1933.889          1–66            1–462                  True
Distribuições de sample_index:
A {0: 168, 1: 168, 2: 168, 3: 168, 4: 168, 5: 168, 6: 168, 7: 168, 8: 168, 9: 168, 10: 168, 11: 168, 12: 168, 13: 168, 14: 168, 15: 168, 16: 168, 17: 

,Sessão,variável,n válido,mínimo,média,mediana,máximo,desvio padrão,amplitude
0,A,T interna (°C),10080,27.8800,31.9308,32.3100,38.6700,2.7189,10.7900
1,A,T externa (°C),10080,26.6900,31.0460,30.9100,35.9900,2.6700,9.3000
2,A,RH interna (%),10080,23.1000,43.4953,44.3800,60.4200,11.2930,37.3200
3,A,RH externa (%),10080,31.4700,47.4938,47.8100,60.4700,8.3165,29.0000
4,A,X (g),10080,-1.1154,0.0064,0.0351,0.6006,0.1512,1.7160
5,A,Y (g),10080,-1.4820,0.0385,0.0858,1.0257,0.4041,2.5077
6,A,Z (g),10080,-0.1365,0.8037,0.8853,1.5327,0.1805,1.6692
7,A,Magnitude (g),10080,0.4885,0.9270,0.9130,1.7120,0.0823,1.2235
8,A,ΔT (°C),10080,-1.4100,0.8848,0.1800,6.1100,1.6125,7.5200
9,A,ΔRH (p.p.),10080,-13.2100,-3.9985,-2.8100,1.0900,3.2801,14.3000


In [5]:
# Consistência com fontes e controles publicados pela auditoria, somente após recálculo independente.
# Relação confirmada no firmware e em todos os fragmentos únicos: seq = 7*(batch_id-1)+frag+1.
for s in SESSIONS:
    u=C[s]["unique"]; relation=(u.seq.astype(int)==7*(u.batch.astype(int)-1)+u.frag.astype(int)+1).all()
    print(s,"relação sequência/batch/fragmento:",relation,"| eventos RECORD no log / CSV:",int((D[s][3].event=="RECORD").sum()),len(D[s][0]),"| BATCH_COMPLETE / batches CSV:",int((D[s][3].event=="BATCH_COMPLETE").sum()),len(D[s][1]))
controls={"A":{"records":10080,"complete":168,"unique":1212,"dup":4,"missing":321,"partial":12,"zero":39},"B":{"records":3960,"complete":66,"unique":462,"dup":0,"missing":0,"partial":0,"zero":0}}
divergences=[]
for s,x in controls.items():
    got={"records":len(D[s][0]),"complete":int((C[s]["batches"].state=="completo").sum()),"unique":len(C[s]["unique"]),"dup":len(C[s]["dup"]),"missing":len(C[s]["missing"]),"partial":int((C[s]["batches"].state=="parcial").sum()),"zero":int((C[s]["batches"].state=="sem fragmento").sum())}
    for k,v in x.items():
        if got[k]!=v: divergences.append(f"{s}: {k} recalculado={got[k]}, controle da auditoria={v}")
print("Divergências:", divergences or "nenhuma")
print("Intervalos entre amostras no mesmo batch (ms):")
for s in SESSIONS: print(s, interval_summary(D[s][0].groupby("batch_id").sensor_ms.diff()))
print("Intervalos entre início de batch (ms):")
for s in SESSIONS: print(s, interval_summary(D[s][1].batch_start_ms.diff()))
print("Distribuição elapsed_ms (primeiros valores/frequências):")
for s in SESSIONS: print(s, D[s][0].elapsed_ms.value_counts().sort_index().head(12).to_dict())

A relação sequência/batch/fragmento: True | eventos RECORD no log / CSV: 10080 10080 | BATCH_COMPLETE / batches CSV: 168 168
B relação sequência/batch/fragmento: True | eventos RECORD no log / CSV: 3960 3960 | BATCH_COMPLETE / batches CSV: 66 66
Divergências: nenhuma
Intervalos entre amostras no mesmo batch (ms):
A {'n': 9912, 'mín': np.float64(488.0), 'mediana': np.float64(500.0), 'máx': np.float64(500.0)}
B {'n': 3894, 'mín': np.float64(86.0), 'mediana': np.float64(500.0), 'máx': np.float64(500.0)}
Intervalos entre início de batch (ms):
A {'n': 167, 'mín': np.float64(29722.0), 'mediana': np.float64(29736.0), 'máx': np.float64(901562.0)}
B {'n': 65, 'mín': np.float64(29721.0), 'mediana': np.float64(29732.0), 'máx': np.float64(29765.0)}
Distribuição elapsed_ms (primeiros valores/frequências):
A {97: 168, 585: 168, 1085: 168, 1585: 168, 2085: 168, 2585: 168, 3085: 168, 3585: 168, 4085: 168, 4585: 168, 5085: 168, 5585: 168}
B {97: 65, 585: 65, 1085: 65, 1585: 65, 2085: 65, 2097: 1, 2183:

In [6]:
# Figuras finais para ambas as sessões. Segmentos preservam lacunas reais entre batches.
plt.rcParams.update({"font.size":9,"axes.labelsize":9,"legend.fontsize":8,"figure.dpi":120,"pdf.fonttype":42,"ps.fonttype":42})
COLORS={"interna":"#0072B2","externa":"#D55E00","delta":"#009E73","mov":"#7A5195"}
def segmented(ax, df, x, y, color, label, gap=0.75, **kw):
    z=df[[x,y,"sensor_ms"]].dropna().sort_values("sensor_ms"); blocks=(z.sensor_ms.diff().fillna(0)/1000>gap).cumsum()
    for i,(_,part) in enumerate(z.groupby(blocks)):
        ax.plot(part[x],part[y],color=color,label=label if i==0 else None,**kw)
def save(fig,name):
    fig.savefig(FIG/f"{name}.png",dpi=360,bbox_inches="tight")
    fig.savefig(FIG/f"{name}.pdf",bbox_inches="tight")
    plt.close(fig)

# Figura 6.1 — termohigrometria, sessões A e B; quatro painéis evitam eixo duplo.
for s in SESSIONS:
    r=D[s][0].copy()
    if THERMAL_CROP_MIN is not None: r=r[r.tempo_min.between(*THERMAL_CROP_MIN)]
    fig,axs=plt.subplots(4,1,sharex=True,figsize=(6.8,7.0),layout="constrained")
    for ax,y,ylab,c1,c2 in [(axs[0],("internal_temp_c","external_temp_c"),"Temperatura (°C)",COLORS["interna"],COLORS["externa"]),(axs[1],("internal_rh_pct","external_rh_pct"),"Umidade relativa (%)",COLORS["interna"],COLORS["externa"])]:
        segmented(ax,r,"tempo_min",y[0],c1,"Interno",lw=1.0); segmented(ax,r,"tempo_min",y[1],c2,"Externo",lw=1.0); ax.set_ylabel(ylab); ax.grid(alpha=.25); ax.legend(ncol=2,loc="best")
    segmented(axs[2],r,"tempo_min","delta_t",COLORS["delta"],"ΔT = interno − externo",lw=1.0); axs[2].set_ylabel("ΔT (°C)"); axs[2].grid(alpha=.25); axs[2].legend(loc="best")
    segmented(axs[3],r,"tempo_min","delta_rh",COLORS["delta"],"ΔRH = interno − externo",lw=1.0); axs[3].set_ylabel("ΔRH (p.p.)"); axs[3].set_xlabel("Tempo de experimento (min)"); axs[3].grid(alpha=.25); axs[3].legend(loc="best")
    save(fig,f"fig_6_1_sessao_{s.lower()}_termohigrometria")

# Figura 6.2 — eixos XYZ e desvio padrão móvel da magnitude, sessões A e B.
for s in SESSIONS:
    r=D[s][0]; fig,axs=plt.subplots(2,1,sharex=True,figsize=(6.8,4.8),layout="constrained")
    for col,lab,c in [("accel_x_g","X",COLORS["interna"]),("accel_y_g","Y",COLORS["externa"]),("accel_z_g","Z",COLORS["delta"])]: segmented(axs[0],r,"tempo_min",col,c,lab,lw=.65)
    axs[0].set_ylabel("Aceleração (g)"); axs[0].grid(alpha=.25); axs[0].legend(ncol=3)
    segmented(axs[1],r,"tempo_min","mag_std_10s",COLORS["mov"],f"Desvio padrão móvel ({WINDOW_SECONDS} s)",lw=.9)
    axs[1].set_ylabel("Desvio padrão da magnitude (g)"); axs[1].set_xlabel("Tempo de experimento (min)"); axs[1].grid(alpha=.25); axs[1].legend(loc="best")
    save(fig,f"fig_6_2_sessao_{s.lower()}_acelerometro")

# Figura 6.3 — comunicação, sessões A e B.
for s in SESSIONS:
    c=C[s]; l=D[s][3]; first=l.received_at.min(); fig,axs=plt.subplots(2,1,figsize=(6.8,5.4),layout="constrained")
    state_color={"completo":"#0072B2","parcial":"#E69F00","sem fragmento":"#999999"}
    for state,part in c["batches"].groupby("state"): axs[0].scatter(part.batch_id,part.unique,s=13,color=state_color[state],label=state.capitalize())
    axs[0].axhline(7,color="black",lw=.8,ls="--",label="Esperado: 7"); axs[0].set_ylabel("Fragmentos únicos por batch"); axs[0].set_xlabel("batch_id"); axs[0].set_ylim(-.4,7.6); axs[0].grid(alpha=.25); axs[0].legend(ncol=2)
    u=c["unique"].copy(); u["min"]= (u.received_at-first).dt.total_seconds()/60; axs[1].scatter(u["min"],np.zeros(len(u)),s=7,color=COLORS["interna"],label="FRAGMENT_RX único")
    for _,g in c["silence"].head(2).iterrows():
        x0=(g.inicio-first).total_seconds()/60; x1=(g.fim-first).total_seconds()/60; axs[1].axvspan(x0,x1,color="#D55E00",alpha=.16); axs[1].text((x0+x1)/2,.06,f"{g.segundos/60:.1f} min",ha="center",va="bottom",fontsize=8)
    axs[1].set_yticks([]); axs[1].set_xlabel("Tempo desde o primeiro evento de recepção (min)"); axs[1].set_ylabel("Eventos"); axs[1].grid(axis="x",alpha=.25); axs[1].legend(loc="upper right")
    save(fig,f"fig_6_3_sessao_{s.lower()}_comunicacao")
print("Figuras salvas em",FIG)

Figuras salvas em /Users/gabriellopesbastos/Projects/BeeCooler/figuras/resultados


In [7]:
# Síntese das métricas de comunicação, validação da magnitude e produção do Markdown.
COMM=[]
for s in SESSIONS:
    c=C[s]; b=c["batches"]; COMM.append({"Sessão":s,"registros":len(D[s][0]),"batches completos":int((b.state=="completo").sum()),"fragmentos únicos":len(c["unique"]),"fragmentos duplicados":len(c["dup"]),"sequence IDs ausentes":len(c["missing"]),"batches parciais":int((b.state=="parcial").sum()),"batches sem fragmento":int((b.state=="sem fragmento").sum()),"cobertura sequence IDs (%)":100*c["coverage"],"cobertura batches completos (%)":100*(b.state=="completo").mean(),"maior silêncio (s)":c["silence"].segundos.max()})
comm=pd.DataFrame(COMM); display(comm.round(3))
for s in SESSIONS:
    r=D[s][0]; tol=(math.sqrt(3)+1)*0.00005+1e-6
    print(s, "erro |magnitude calculada − armazenada| máximo:",fmt(r.mag_error.abs().max(),6),"g; tolerância conservadora:",fmt(tol,6),"g; dentro:",bool((r.mag_error.abs()<=tol).all()))

def md_table(df, digits=3):
    x=df.copy()
    for c in x.select_dtypes("number"): x[c]=x[c].map(lambda v: fmt(v,digits))
    return x.to_markdown(index=False)
A,B=COMM[0],COMM[1]
qa=quality.copy(); qa["início recepção"]=qa["início recepção"].astype(str); qa["fim recepção"]=qa["fim recepção"].astype(str)
lines=["# AP1 — resultados da análise quantitativa da Fase 0","", "## Metodologia", "", "Foram lidos diretamente `data/records.csv`, `data/batches.csv`, `data/gateway_stats.csv` e os dois logs brutos de gateway. Cada cálculo filtrou explicitamente o `session_id`. As curvas de sensores usam `sensor_ms` e `tempo_min = (sensor_ms − primeiro sensor_ms)/60000`; a comunicação usa o horário `received_at` dos eventos de log. Não houve interpolação entre batches ausentes.", "", "A integridade dos cinco arquivos foi conferida por SHA-256 contra a auditoria; todos os hashes coincidiram. Os controles independentes da auditoria também foram reproduzidos sem divergências." if not divergences else "**Divergências frente aos controles da auditoria:** " + "; ".join(divergences), "", "## Qualidade dos dados", "", md_table(qa[["Sessão","registros","batches CSV","batches com 60","NaN sensores","duplicatas de registro","duração sensor (s)","duração recepção (s)","batch min–max","sequence min–max","sensor=start+elapsed"]]), "", "A distribuição de `sample_index` foi uniforme de 0 a 59 em todos os batches. A igualdade `sensor_ms = batch_start_ms + elapsed_ms` foi satisfeita para todas as linhas. Na sessão B, a primeira amostra manteve `T interna` e `T externa` como NaN; as demais variáveis dessa linha foram preservadas.", "", "## Estatísticas dos sensores", "", "As estatísticas seguintes usam somente valores válidos; o desvio padrão é amostral. Os deltas representam diferença entre condições medidas e não erro entre sensores.", "", md_table(sensor_stats,4), "", "## Comunicação", "", "A relação `sequence_id = 7 × (batch_id − 1) + fragment_index + 1` foi confirmada nos fragmentos únicos de ambas as sessões. Dentro do intervalo observado de batches, cada batch foi classificado com sete fragmentos esperados, 1–6 como parcial e zero como sem fragmento. A sessão A começa no batch 16; batches anteriores não foram classificados como perdas.", "", md_table(comm,3), ""]
for s in SESSIONS:
    c=C[s]; top=c["silence"].head(3).copy(); top["início"]=top.inicio.astype(str); top["fim"]=top.fim.astype(str); lines += [f"### Intervalos sem FRAGMENT_RX — sessão {s}","",md_table(top[["início","fim","segundos"]],3),""]
lines += ["## Figuras candidatas ao relatório", "", "- `figuras/resultados/fig_6_1_sessao_a_termohigrometria.{pdf,png}` e `fig_6_1_sessao_b_termohigrometria.{pdf,png}` — evolução de temperatura, umidade e deltas; quatro painéis evitam eixo duplo.", "- `figuras/resultados/fig_6_2_sessao_a_acelerometro.{pdf,png}` e `fig_6_2_sessao_b_acelerometro.{pdf,png}` — eixos XYZ e desvio padrão móvel de 10 s da magnitude.", "- `figuras/resultados/fig_6_3_sessao_a_comunicacao.{pdf,png}` e `fig_6_3_sessao_b_comunicacao.{pdf,png}` — fragmentos únicos por batch e eventos de recepção; os dois maiores intervalos são destacados objetivamente.", "", "## Interpretação factual", "", "Na sessão B, os 66 batches completos materializaram 3.960 registros. No intervalo observado, os 462 fragmentos únicos cobriram todos os sequence IDs de 1 a 462, sem duplicatas, batches parciais ou batches sem fragmento (Tabela de comunicação).", "", "As Figuras 6.1 das sessões A e B mostram trechos de valores próximos entre sensores e outros com divergência pronunciada; em B, ΔT variou de " + fmt(sensor_stats.query("Sessão=='B' and variável=='ΔT (°C)'").iloc[0]["mínimo"],2) + " a " + fmt(sensor_stats.query("Sessão=='B' and variável=='ΔT (°C)'").iloc[0]["máximo"],2) + " °C e ΔRH de " + fmt(sensor_stats.query("Sessão=='B' and variável=='ΔRH (p.p.)'").iloc[0]["mínimo"],2) + " a " + fmt(sensor_stats.query("Sessão=='B' and variável=='ΔRH (p.p.)'").iloc[0]["máximo"],2) + " p.p. (estatísticas recalculadas em `records.csv`). O padrão é compatível com o relato de condições distintas durante parte do ensaio, sem atribuir horários a intervenções humanas não registradas.", "", "As Figuras 6.2 mostram os eixos XYZ e a métrica de variação local em ambas as sessões; a sessão A apresenta a maior faixa observada. A magnitude recalculada de X, Y e Z concordou com o campo serializado dentro da tolerância de quantização; o painel inferior apresenta desvio padrão móvel de 10 s, aproximadamente 20 amostras, como medida de variação local da magnitude. Ela evidencia trechos relativamente estáveis e outros de maior perturbação mecânica, mas não mede frequência de vibração nem permite caracterização espectral.", "", "Na sessão A foram observados " + str(A["fragmentos únicos"]) + " fragmentos únicos entre os sequence IDs " + str(C['A']['seqlo']) + " e " + str(C['A']['seqhi']) + ", com " + str(A["sequence IDs ausentes"]) + " IDs ausentes e cobertura de " + fmt(A["cobertura sequence IDs (%)"],2) + "%. Dos " + str(len(C['A']['batches'])) + " batches no intervalo observado (16–234), " + str(A["batches completos"]) + " foram completos, " + str(A["batches parciais"]) + " parciais e " + str(A["batches sem fragmento"]) + " sem fragmentos. A Figura 6.3 da sessão A mostra recuperação de recepção após silêncios de " + fmt(C['A']['silence'].iloc[0].segundos/60,2) + " e " + fmt(C['A']['silence'].iloc[1].segundos/60,2) + " min. Esses intervalos ocorreram no ensaio de campo e são compatíveis com a perda de enlace relatada pela equipe, mas os logs não identificam a causa física.", "", "## Limitações", "", "`ACK_TX` significa que o gateway solicitou/enviou ACK pela API; não prova que o nó o recebeu, nem confirma remontagem completa ou persistência no CSV. Não há logs do transmissor com `FRAGMENT_TX`, resultados de tentativa ou resumo de batch; portanto não foram calculadas taxa real por tentativa, retransmissões totais nem RTT de ACK. O ADXL345 foi configurado com ODR de 100 Hz, mas os CSVs têm aproximadamente uma leitura a cada 500 ms; FFT, PSD e inferência sobre frequências de vibração não são suportadas por estes dados.", "", "## Proposta factual para 6.1 — O sistema em operação", "", "O fluxo da Fase 0 registrou amostras no nó, agrupou 60 registros por batch e materializou no coletor os batches completos recebidos pelo gateway. A validação de `sensor_ms = batch_start_ms + elapsed_ms` e a distribuição de `sample_index` de 0 a 59 confirmam a estrutura de 60 amostras por batch nas duas sessões (Tabela de qualidade).", "", "No ensaio de 16/09 (sessão B), a série temporal térmica e higrométrica apresenta tanto períodos de acompanhamento como períodos de divergência entre os sensores (Figura 1). Foram preservados os dois NaN da primeira linha de temperatura; as estatísticas foram calculadas com 3.959 valores válidos para cada temperatura e 3.960 para umidade e deltas, conforme a tabela de estatísticas.", "", "O acelerômetro respondeu às perturbações do conjunto em escala de amostras de aproximadamente 500 ms. As Figuras 6.2 apresentam eixos XYZ e uma medida local de dispersão da magnitude em janela de 10 s; não é uma análise de vibração espectral.", "", "A comunicação foi contínua na sessão B para o intervalo observado: 66 batches completos, 462 fragmentos únicos, cobertura de 100% dos sequence IDs e nenhuma duplicata ou batch parcial (Tabela de comunicação).", "", "No ensaio de 04/09 (sessão A), a comunicação não manteve a mesma cobertura. A Figura 6.3 da sessão A e a tabela mostram 168 batches completos, 12 parciais e 39 sem fragmentos no intervalo de batches 16–234, além de 321 sequence IDs ausentes. Após os dois maiores períodos sem `FRAGMENT_RX`, de 901,391 s e 571,323 s, o gateway voltou a registrar fragmentos, evidenciando recuperação observada do enlace.", "", "## Evidências disponíveis para 6.2", "", "| Capacidade observada | Evidência | Métrica | Limitação |", "|---|---|---|---|", "| Aquisição por dois SHT30 | CSV de registros e Figura 1 | 3.960 registros B; temperaturas e RH materializados | condições deliberadamente diferentes não medem erro entre sensores |", "| Aquisição do ADXL345 | CSV e Figura 2 | XYZ, magnitude e dispersão móvel disponíveis | amostragem exportada ≈ 2 Hz; sem análise espectral |", "| Transporte e persistência de dados recebidos | logs, CSVs e tabela de comunicação | 66 batches completos / 3.960 registros em B | CSV representa batches completados pelo gateway |", "| ACK e retransmissão limitada implementados | evidência de ACK_TX no log e auditoria de código | ACK_TX: 462 em B; 1.216 em A | o log não comprova recepção do ACK pelo nó ou número de tentativas |", "", "## Pontos de discussão para 6.3", "", "- Os dois SHT30 e o acelerômetro foram adquiridos e exportados pelo protótipo.", "- Os dados recebidos em batches completos foram transportados ao gateway e persistidos pelo coletor.", "- O firmware emprega ACK por fragmento e retransmissão limitada, mas descarta o batch após o ciclo de transmissão mesmo sem entrega completa; não há persistência confirmada de batches não entregues.", "- A sessão B e a sessão A apresentam contraste mensurável de cobertura: 100% de sequence IDs observados em B e " + fmt(A["cobertura sequence IDs (%)"],2) + "% em A.", "- A sessão A contém perda e posterior recuperação de recepção no cenário de campo; a causa física requer evidência adicional.", "- Os CSVs não suportam caracterização espectral do acelerômetro.", "- A confirmação de requisitos das Seções 2 e 5 permanece necessária antes de declarar requisitos atendidos."]
DOC.write_text("\n".join(lines)+"\n")
assert HASH_BEFORE == {k:sha256(v) for k,v in INPUTS.items()}, "Arquivo de entrada foi modificado"
print("Markdown salvo:",DOC)
print("Resumo:",comm.to_dict(orient="records"))

,Sessão,registros,batches completos,fragmentos únicos,fragmentos duplicados,sequence IDs ausentes,batches parciais,batches sem fragmento,cobertura sequence IDs (%),cobertura batches completos (%),maior silêncio (s)
0,A,10080,168,1212,4,321,12,39,79.061,76.712,901.391
1,B,3960,66,462,0,0,0,0,100.000,100.000,29.825


A erro |magnitude calculada − armazenada| máximo: 0.000050 g; tolerância conservadora: 0.000138 g; dentro: True
B erro |magnitude calculada − armazenada| máximo: 0.000050 g; tolerância conservadora: 0.000138 g; dentro: True
Markdown salvo: /Users/gabriellopesbastos/Projects/BeeCooler/docs/AP1_RESULTADOS_ANALISE.md
Resumo: [{'Sessão': 'A', 'registros': 10080, 'batches completos': 168, 'fragmentos únicos': 1212, 'fragmentos duplicados': 4, 'sequence IDs ausentes': 321, 'batches parciais': 12, 'batches sem fragmento': 39, 'cobertura sequence IDs (%)': 79.06066536203522, 'cobertura batches completos (%)': 76.71232876712328, 'maior silêncio (s)': 901.391}, {'Sessão': 'B', 'registros': 3960, 'batches completos': 66, 'fragmentos únicos': 462, 'fragmentos duplicados': 0, 'sequence IDs ausentes': 0, 'batches parciais': 0, 'batches sem fragmento': 0, 'cobertura sequence IDs (%)': 100.0, 'cobertura batches completos (%)': 100.0, 'maior silêncio (s)': 29.825}]
